<a href="https://colab.research.google.com/github/columbia-data-club/meetings/blob/main/2026/september_23_python_and_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![A blue background with the Hugging Face logo and the words Columbia Data Club on it](https://raw.githubusercontent.com/columbia-data-club/meetings/refs/heads/main/assets/images/bede%20hugging%20face.png)

# Python AI with Hugging Face

September 23, 2026

by [Moacir P. de Sá Pereira](https://moacir.com) for the [Columbia Data Club](https://github.com/columbia-data-club/)


## What Is Hugging Face and Why Does It Have a Dumb Name?

Hugging Face ([http://huggingface.co](http://huggingface.co)) is a platform dedicated to machine learning. As sort of the “GitHub of Machine Learning,” HF lets people interested in machine learning publish their models, along with their datasets, when needed.

The dumb name comes from its logo, the 🤗 emoji. It was meant as a joke and placeholder, but then it stuck.

In addition to a website, however, HF also provides a few important Python libraries, but we are focusing on one today, [`transformers`](https://pypi.org/project/transformers/), a standardization library for letting researchers like us build a general machine learning framework that is model agnostic.

This is the important, secret sauce that makes up the HF one-two punch:

1. All the models in one place.
1. One way to activate them all.

Hugging Face’s popularity is maintained by its versioning (important for reproducibility) and its popularity among labs keeping the model inventory fresh and cutting edge.

## Get Started with Hugging Face

1. Create a free account at [Hugging Face](https://huggingface.co).
1. Go to Settings > Access Tokens
1. Click on “Create new token”
1. Click on ”Read” since you only need a token for downloading
1. Give it a name (like `data-club-token`) and click “Create token”
1. Copy the token and paste it somewhere useful or keep it in the clipboard
1. Click on the key icon on the left of this page, which opens Colab secrets
1. Click “Add new secret”
1. For name, **type** `HF_TOKEN` (don’t copy it!)
1. For value, paste in the token
1. Toggle “Notebook access.”

We're close to done, but there's one more step:

1. Click on the down arrow in the top right beside “RAM” and “Disk” and change the runtime type to T4 GPU.

In [1]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

Now we have an API key saved with Hugging Face, which will let us download our models with greater facility.

## Three Simple Examples

For our next step, let’s go with three simple examples that leverage the three-lines-of-code ethos that makes `transformers` so appealing to researchers who just need to do inference, who just need models to do something for them.

These will make use of the [`pipeline()`](https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines) abstraction.

### Text Mining with `"text-generation"`

For our first example, we'll use the [text generation](https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines#transformers.TextGenerationPipeline) pipeline, which works like a chatbot.

For our model, we’ll use the [`Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) model. Each part of the name gives us a little bit of information about the model:

* `Qwen` means the model comes from [Qwen](https://qwen.ai/), Alibaba’s AI lab.
* `2.5` means it’s version 2.5, which was released two years ago. Newer models are available.
* `1.5B` means the model has 1.5 billion parameters. Qwen has models with _trillions_ of parameters. But the free tier of Google Colab won’t let us run those kinds of models.
* `Instruct` is a special type of model that is designed to take instructions, which is exactly what we are about to do.

In [2]:
from transformers import pipeline
import torch
torch.cuda.empty_cache()

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
# device = 0 forces us to use the GPU.
pipe = pipeline("text-generation", model=model_id, device=0)

abstract = """
Introduction
This study examines how inpatients with catatonia responded when treated
with or without antipsychotic medications. It employs two metrics to
account for both the catatonic symptoms and the more traditionally
psychotic symptoms. Length of stay is a secondary metric.

Methods
The primary investigator retrospectively collected data on 164 patients
diagnosed with catatonia on an academic inpatient service from July 2018
through September 2023. The treatments of these patients were separated
into two distinct categories: Group A (n = 81) received antipsychotic
medication with benzodiazepines and/or electroconvulsive therapy, and
Group B (n = 83) received benzodiazepines and/or electroconvulsive therapy
without antipsychotics. Scores on admission from the Bush Francis Catatonia
Rating Scale and positive symptom subscale from the Positive and Negative
Syndrome Scale were collected and compared with scores at discharge. ANOVA
analysis was used to compare outcomes.

Results
The antipsychotic-treated group demonstrated significantly higher Bush
Francis Catatonia Rating Scale scores (p < 0.0001) and positive scale of
the Positive and Negative Syndrome Scale scores (p < 0.0001) at discharge
than the group receiving only benzodiazepines and/or electroconvulsive
therapy. Patients hospitalized multiple times and treated under both
conditions were used as their own controls. Bush Francis Catatonia Rating
Scale scores (p < 0.0001) and positive scale of the Positive and Negative
Syndrome Scale scores (p < 0.0001) were higher at discharge when
antipsychotics were added to treatment.

Conclusion
Patients treated without antipsychotics showed greater improvement in both
their traditional catatonic symptoms and in their psychotic symptoms.
"""

messages = [{"role": "user", "content": f"""
Extract the following fields from the abstract and return ONLY a JSON object:
- Study_Design
- Sample_Size
- Methodology
- Key_Metric_and_Value

Abstract: {abstract}
"""}]

prompt_text = pipe.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
outputs = pipe(
    prompt_text,
    max_new_tokens=150,
    temperature=0.1,
    do_sample=True
)
print(outputs[0]["generated_text"][len(prompt_text):])
del pipe

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


```json
{
  "Study_Design": "Retrospective",
  "Sample_Size": {
    "Total_Patients": 164,
    "Antipsychotic_Group": 81,
    "Benzodiazepine_Electroconvulsive_Treatment_Group": 83
  },
  "Methodology": [
    "ANOV analysis"
  ],
  "Key_Metric_and_Value": {
    "Bush Francis Catatonia Rating Scale Scores": {
      "Antipsychotic Group": "Significantly higher",
      "Benzodiazepine_Electroconvulsive_Treatment Group": "Lower"
    },
    "Positive Scale of the Positive and Negative Syndrome Scale Scores


We can then use just this chunk to turn it into actual json and feed it into a database or do whatever we want to with it. Hopefully the iterative potential is clear. We create a list of abstracts, reinitialize `messages`, tokenize the messages, and then pipe them into our model.

### Scanned Document Analysis with `image-text-to-text`

Next, we can use similar code to do image analysis as well. Now we’ll be using [`Qwen2-VL-2B-Instruct`](https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct). `VL` here stands for “visual language,” and the VL models are multimodal, in the sense that they understand text and images.

The pipeline is [`image-text-to-text`](https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines#transformers.ImageTextToTextPipeline).

We’ll be feeding in this page of text from a book I’m using in my own research:

![page from Laperousse](https://github.com/columbia-data-club/meetings/blob/main/assets/images/laperousse-page.png?raw=true)

In [19]:
torch.cuda.empty_cache()

model_id = "Qwen/Qwen2-VL-2B-Instruct"
pipe = pipeline("image-text-to-text", model=model_id, device=0)

IMAGE_URL = "https://raw.githubusercontent.com/columbia-data-club/meetings/refs/heads/main/assets/images/laperousse-page.png"

prompt = """
This is a scanned page from a document from the late 17th or early 18th century.

1. Transcribe the full text verbatim.
2. Extract any dates, personal names, and place names as a JSON object.
"""

messages = [{"role": "user", "content": [
    {"type": "image", "url": IMAGE_URL},
    {"type": "text", "text": prompt,}
]}]

outputs = pipe(text=messages, max_new_tokens=512)

print(outputs[0]["generated_text"][1]["content"])

del pipe

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "dates": [
    {
      "year": 1787
    }
  ],
  "personal_names": [
    {
      "name": "toyon",
      "title": "chief of the village"
    },
    {
      "name": "several others",
      "title": "of it's inhabitants"
    }
  ],
  "place_names": [
    {
      "name": "Bay of Avatscha"
    }
  ]
}
```


### Categorizing Research with `zero-shot-classification`

For our final trick, we’ll do some on-the-fly (“zero-shot”) classification, where we rely on the model’s _already existing_ sense of the world to classify texts. That is, we are not producing a training dataset to show how things should be classified ahead of time. Nor are our categories even predetermined. We can send our list of categories along with the query itself. This is the [`zero-shot-classification`](https://huggingface.co/docs/transformers/v5.17.0/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline) pipeline.

For our model, we’re using Facebook’s [`bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli). BART stands for “Bidirectional and Auto-Regressive Transformers.” That means it reads sentences from both directions at once and predicts the next word from what has already happened. The MNLI part is “[Multi-Genre Natural Language Inference](https://cims.nyu.edu/~sbowman/multinli/),” where for each string of text, the model tests three hypotheses against each possible category.



In [2]:
torch.cuda.empty_cache()

model_id = "facebook/bart-large-mnli"
pipe = pipeline("zero-shot-classification", model=model_id, device=0)
abstracts = [
    "We develop a novel theoretical framework for understanding the relationship between social media use and political polarization, drawing on communication theory and social identity theory.",
    "Using a dataset of 10,000 survey responses and multivariate regression analysis, we find a significant positive correlation between social media usage and political extremism (β=0.34, p<0.001).",
    "We propose three policy recommendations for regulating social media platforms: (1) mandatory algorithmic transparency, (2) age verification requirements, and (3) independent oversight boards.",
    "Our methodology combines computational text analysis with qualitative interviews. We use LDA topic modeling to identify themes in 50,000 tweets, followed by semi-structured interviews with 25 participants."
]

candidate_labels = [
    "theoretical framework",
    "empirical results",
    "policy recommendations",
    "methodology"
]

print("Zero-Shot Classification Results:\n")
for i, abstract in enumerate(abstracts, 1):
    result = pipe(abstract, candidate_labels)

    top_label = result['labels'][0]
    top_score = result['scores'][0]

    print(f"Abstract {i}:")
    print(f"  Text: {abstract[:100]}...")
    print(f"  Predicted category: {top_label} (confidence: {top_score:.3f})")
    print()

del pipe

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Zero-Shot Classification Results:

Abstract 1:
  Text: We develop a novel theoretical framework for understanding the relationship between social media use...
  Predicted category: theoretical framework (confidence: 0.945)

Abstract 2:
  Text: Using a dataset of 10,000 survey responses and multivariate regression analysis, we find a significa...
  Predicted category: empirical results (confidence: 0.495)

Abstract 3:
  Text: We propose three policy recommendations for regulating social media platforms: (1) mandatory algorit...
  Predicted category: policy recommendations (confidence: 0.874)

Abstract 4:
  Text: Our methodology combines computational text analysis with qualitative interviews. We use LDA topic m...
  Predicted category: methodology (confidence: 0.912)

